# XGBoost with Frequency Encoding, Target Encoding, and Seed Ensemble

This notebook trains an XGBoost binary classification model using a feature set built from the original columns, categorical-style numeric features, digit-based features, frequency encoding, and target encoding.

The main idea is to enrich the baseline feature set with both **count-based signals** and **target-based categorical interactions**, while keeping the modeling pipeline relatively simple and robust.

## Feature Engineering

The model uses the following groups of features:

- Original baseline features
- Numeric features treated as categorical features
- Digit-based features
- Single-column frequency encoding
- Pairwise joint frequency encoding
- Single-column target encoding
- Pairwise target encoding

For the pairwise features, combinations are generated from key racing-related columns such as:

- `Driver`
- `Compound`
- `Race`
- `Year`
- `PitStop`
- `LapNumber`
- `Stint`
- `TyreLife`
- `Position`
- `RaceProgress`
- `Position_Change`

These pairwise features are useful because pit stop behavior is likely influenced not only by individual variables, but also by interactions such as driver × compound, race × lap number, stint × tyre life, and position × race progress.

## Target Encoding

Target encoding is applied using RAPIDS cuML's `TargetEncoder`.

In particular, this notebook uses cuML for both single-column and pairwise target encoding, which helps make the encoding step faster on GPU.  
The target encoding is performed with an inner cross-validation scheme to reduce leakage from the training labels.

The main configuration is:

- Outer CV: `5` folds
- Target encoding inner CV: `5` folds
- Target encoding smoothing: `20`

## Original Data Augmentation

In each fold, the training split is concatenated with the original dataset before fitting the feature encoders and the model.

The validation fold is always kept separate, so the out-of-fold evaluation is still based only on the competition training data.

## Model

The final model is based on `XGBClassifier` with GPU training.

The main model settings are:

- Objective: `binary:logistic`
- Evaluation metric: `auc`
- Tree method: `hist`
- Device: `cuda`
- Learning rate: `0.01`
- Max depth: `9`
- Early stopping: `200` rounds

## Seed Ensemble

To improve prediction stability, this notebook trains multiple XGBoost models with different random seeds inside each fold.

For each outer fold, 5 models are trained using different seeds, and their validation/test predictions are averaged.

This gives:

- `5` outer folds
- `5` seeds per fold
- `25` total XGBoost models

The final out-of-fold prediction is the average of the seed ensemble for each validation fold.  
The final test prediction is averaged across all folds and all seeds.

This seed ensemble is expected to reduce variance and make the final submission more stable than relying on a single random seed.

## Note

The implementation and documentation of this notebook were organized with the assistance of GPT-5.5.  
All modeling choices, feature engineering ideas, validation design, and final experiments were reviewed and adjusted manually.

In [ ]:
import warnings
warnings.simplefilter('ignore')

# Load Data

In [ ]:
import pandas as pd, numpy as np, os

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e5/test.csv')
orig = pd.read_csv('/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv')
print('Train Shape:', train.shape)
display(train.head(3))
print('Test Shape:', test.shape)
display(test.head(3))
print('Orig Shape:', orig.shape)
display(orig.head(3))

In [ ]:
TARGET = 'PitNextLap'
BASE = [col for col in train.columns if col not in ['id', TARGET]]
CATS = [col for col in BASE if train[col].dtype == 'object']

print(len(BASE), 'Baseline Features.')
print(len(CATS), 'Categorical Features.')
print("Categorical Columns:", CATS)

for col in CATS:
    combined = pd.concat(
        [
            train[col].astype(str),
            test[col].astype(str),
            orig[col].astype(str)
        ],
        axis=0
    )

    uniques = combined.unique()
    mapping = {v: i for i, v in enumerate(uniques)}

    train[col] = train[col].astype(str).map(mapping).astype("int32").astype("category")
    test[col] = test[col].astype(str).map(mapping).astype("int32").astype("category")
    orig[col] = orig[col].astype(str).map(mapping).astype("int32").astype("category")

# Feature Engineering

## NUM as CAT

In [ ]:
NUM_as_CAT = []

# =========================
# Num -> Cat exact features
# =========================
NUM_CAT_BASE = [
    'LapTime (s)',
    'LapTime_Delta',
    'Cumulative_Degradation'
]

for c in NUM_CAT_BASE:
    new_col = f'{c}_cat'
    for df in [train, test, orig]:
        df[new_col] = df[c].astype(str)
    NUM_as_CAT.append(new_col)

print(len(NUM_as_CAT), 'Exact Num->Cat Features Created!')


# =========================
# Rounded Num -> Cat features
# =========================
ROUND_CONFIG = {
    # LapTime itself is around 70-100 sec for normal laps,
    # so 0.1 / 0.5 / 1 sec bins are likely useful.
    'LapTime (s)': {
        'round_digits': [1, 0],
        'round_steps': [0.5, 1.0, 2.0, 5.0],
    },

    # Delta has large outliers, but most values are around -10 to 1.
    # Fine bins around 0.1/0.5/1 and rough bins are both worth trying.
    'LapTime_Delta': {
        'round_digits': [1, 0],
        'round_steps': [0.5, 1.0, 2.0, 5.0, 10.0],
    },

    # Cumulative degradation has wider scale.
    # 1 / 2 / 5 / 10 sec style bins are likely more meaningful.
    'Cumulative_Degradation': {
        'round_digits': [1, 0],
        'round_steps': [1.0, 2.0, 5.0, 10.0, 20.0],
    },
}


def round_to_step(s, step):
    return np.round(s / step) * step


for c, cfg in ROUND_CONFIG.items():

    # Standard decimal rounding
    for d in cfg['round_digits']:
        new_col = f'{c}_round{d}_cat'
        for df in [train, test, orig]:
            df[new_col] = df[c].round(d).astype(str)
        NUM_as_CAT.append(new_col)

    # Step-wise rounding / binning
    for step in cfg['round_steps']:
        step_name = str(step).replace('.', 'p')
        new_col = f'{c}_round_step_{step_name}_cat'

        for df in [train, test, orig]:
            df[new_col] = round_to_step(df[c], step).astype(str)

        NUM_as_CAT.append(new_col)


print(len(NUM_as_CAT), 'Total Num->Cat Features Created!')
NUM_as_CAT

## DIGIT

In [ ]:
DIGIT_FEATURES = []

DIGIT_BASE = [
    'Year',
    'PitStop',
    'LapNumber',
    'Stint',
    'TyreLife',
    'Position',
    'LapTime (s)',
    'LapTime_Delta',
    'Cumulative_Degradation',
    'RaceProgress',
    'Position_Change',
]

DECIMAL_DIGIT_BASE = [
    'LapTime (s)',
    'LapTime_Delta',
    'Cumulative_Degradation',
    'RaceProgress',
]

INT_POSITIONS = [1, 10, 100, 1000]
DECIMAL_POSITIONS = [1, 2, 3]


def safe_colname(c):
    return (
        c.replace(' ', '_')
         .replace('(', '')
         .replace(')', '')
         .replace('/', '_')
         .replace('-', '_')
    )


def to_numeric_array(s):
    x = pd.to_numeric(s, errors='coerce').astype(float).values
    x = np.round(x, 6) 
    return x


# =========================
# Integer Digit Features
# =========================

for c in DIGIT_BASE:
    if not all(c in df.columns for df in [train, test, orig]):
        print(f"[Skip] {c} is not found in all dataframes.")
        continue

    sc = safe_colname(c)

    sign_col = f'{sc}_sign'
    for df in [train, test, orig]:
        x = to_numeric_array(df[c])
        sign = np.sign(np.nan_to_num(x, nan=0.0)).astype(np.int8)
        df[sign_col] = sign
    DIGIT_FEATURES.append(sign_col)

    for p in INT_POSITIONS:
        nc = f'{sc}_digit_{p}s'

        for df in [train, test, orig]:
            x = to_numeric_array(df[c])
            x_abs = np.abs(np.nan_to_num(x, nan=0.0))

            int_part = np.floor(x_abs).astype(np.int64)
            digit = ((int_part // p) % 10).astype(np.int8)

            df[nc] = digit

        DIGIT_FEATURES.append(nc)


# =========================
# Decimal Digit Features
# =========================

for c in DECIMAL_DIGIT_BASE:
    if not all(c in df.columns for df in [train, test, orig]):
        print(f"[Skip] {c} is not found in all dataframes.")
        continue

    sc = safe_colname(c)

    for d in DECIMAL_POSITIONS:
        nc = f'{sc}_decimal_digit_{d}'

        for df in [train, test, orig]:
            x = to_numeric_array(df[c])
            x_abs = np.abs(np.nan_to_num(x, nan=0.0))
            digit = (np.floor(x_abs * (10 ** d)).astype(np.int64) % 10).astype(np.int8)

            df[nc] = digit

        DIGIT_FEATURES.append(nc)


print(len(DIGIT_FEATURES), 'DIGIT Features Created!')
print(DIGIT_FEATURES[:20])

# Model

In [ ]:
from itertools import combinations
import cudf

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from cuml.preprocessing import TargetEncoder as cuTargetEncoder
from xgboost import XGBClassifier

N_SPLITS = 5
N_TE_FOLDS = 5
TE_SMOOTH = 20

SEEDS = [42, 43, 44, 45, 46]

LOSSGUIDE_MAX_LEAVES = 64

TE_BASE = [
    'Driver', 'Compound', 'Race', 'Year', 'PitStop',
    'LapNumber', 'Stint', 'TyreLife', 'Position',
    'RaceProgress', 'Position_Change',
]

BIGRAM_SPECS = list(combinations(TE_BASE, 2))
FEATURES = BASE + NUM_as_CAT + DIGIT_FEATURES

print(len(BIGRAM_SPECS), "BIGRAM specs")
print(len(FEATURES), "Base features")


oof_preds = np.zeros(len(train), dtype=np.float32)
test_preds = np.zeros(len(test), dtype=np.float32)

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=42
)

X_orig = orig[FEATURES].copy()
y_orig = orig[TARGET].copy().reset_index(drop=True)


def to_numpy(x):
    if hasattr(x, "get"):
        return x.get()
    return np.asarray(x)


def make_inner_fold_ids(y, n_splits=7, seed=42):
    fold_ids = np.zeros(len(y), dtype=np.int32)

    inner = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed
    )

    dummy_X = np.zeros((len(y), 1))

    for fold, (_, va_idx) in enumerate(inner.split(dummy_X, y)):
        fold_ids[va_idx] = fold

    return cudf.Series(fold_ids)


def add_frequency_encode(
    X_tr,
    X_va,
    X_test,
    cols,
    out_col=None,
    normalize=True,
):
    cols = list(cols)

    if out_col is None:
        out_col = "fe__" + "__".join(cols)

    X_tr_key = X_tr[cols].astype(str)
    X_va_key = X_va[cols].astype(str)
    X_test_key = X_test[cols].astype(str)

    denom = len(X_tr) if normalize else 1.0

    if len(cols) == 1:
        c = cols[0]
        freq_map = X_tr_key[c].value_counts(dropna=False)

        X_tr[out_col] = (
            X_tr_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

        X_va[out_col] = (
            X_va_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

        X_test[out_col] = (
            X_test_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

    else:
        freq_df = (
            X_tr_key
            .groupby(cols, dropna=False)
            .size()
            .reset_index(name=out_col)
        )

        def map_joint_freq(df_key):
            return (
                df_key
                .reset_index(drop=True)
                .merge(freq_df, on=cols, how="left")[out_col]
                .fillna(0)
                .astype(np.float32)
                .values / denom
            )

        X_tr[out_col] = map_joint_freq(X_tr_key)
        X_va[out_col] = map_joint_freq(X_va_key)
        X_test[out_col] = map_joint_freq(X_test_key)


def add_cuml_te(
    X_tr,
    X_va,
    X_test,
    X_tr_g,
    X_va_g,
    X_test_g,
    y_tr_g,
    cols,
    out_col,
    fold_ids_g,
    seed=42,
    smooth=20,
    n_folds=7,
):
    cols = list(cols)

    te = cuTargetEncoder(
        n_folds=n_folds,
        smooth=smooth,
        seed=seed,
        split_method="random",
        output_type="numpy",
        stat="mean",
        multi_feature_mode="combination",
    )

    tr_enc = te.fit_transform(
        X_tr_g[cols],
        y_tr_g,
        fold_ids=fold_ids_g
    )

    va_enc = te.transform(X_va_g[cols])
    test_enc = te.transform(X_test_g[cols])

    X_tr[out_col] = to_numpy(tr_enc).reshape(-1).astype(np.float32)
    X_va[out_col] = to_numpy(va_enc).reshape(-1).astype(np.float32)
    X_test[out_col] = to_numpy(test_enc).reshape(-1).astype(np.float32)


for fold, (tr_idx, va_idx) in enumerate(skf.split(train[FEATURES], train[TARGET])):
    print(f"\nFold {fold + 1}/{N_SPLITS}")

    X_tr_base = train[FEATURES].iloc[tr_idx].copy()
    y_tr_base = train[TARGET].iloc[tr_idx].copy().reset_index(drop=True)

    X_va = train[FEATURES].iloc[va_idx].copy()
    y_va = train[TARGET].iloc[va_idx].copy()

    X_test = test[FEATURES].copy()

    X_tr = pd.concat(
        [X_tr_base.reset_index(drop=True), X_orig.reset_index(drop=True)],
        axis=0,
        ignore_index=True
    )

    y_tr = pd.concat(
        [y_tr_base, y_orig],
        axis=0,
        ignore_index=True
    )

    print("Base:", X_tr.shape, X_va.shape, X_test.shape)

    te_source_cols = sorted(set(['Driver'] + NUM_as_CAT + TE_BASE))
    te_source_cols = [c for c in te_source_cols if c in X_tr.columns]

    X_tr_g = cudf.from_pandas(X_tr[te_source_cols].astype(str))
    X_va_g = cudf.from_pandas(X_va[te_source_cols].astype(str))
    X_test_g = cudf.from_pandas(X_test[te_source_cols].astype(str))
    y_tr_g = cudf.Series(y_tr.values)

    te_seed = 1000 + fold

    fold_ids_g = make_inner_fold_ids(
        y_tr,
        n_splits=N_TE_FOLDS,
        seed=te_seed
    )

    FE_SINGLE_COLS = sorted(set(['Driver'] + NUM_as_CAT))
    FE_SINGLE_COLS = [c for c in FE_SINGLE_COLS if c in X_tr.columns]

    for c in FE_SINGLE_COLS:
        add_frequency_encode(
            X_tr,
            X_va,
            X_test,
            cols=(c,),
            out_col=f"fe__{c}",
            normalize=True,
        )

    for cols in BIGRAM_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_frequency_encode(
                X_tr,
                X_va,
                X_test,
                cols=cols,
                out_col="fe2__" + "__".join(cols),
                normalize=True,
            )

    print("After FE:", X_tr.shape, X_va.shape, X_test.shape)

    if 'Driver' in X_tr.columns:
        add_cuml_te(
            X_tr,
            X_va,
            X_test,
            X_tr_g,
            X_va_g,
            X_test_g,
            y_tr_g,
            cols=('Driver',),
            out_col='te_Driver',
            fold_ids_g=fold_ids_g,
            seed=te_seed,
            smooth=TE_SMOOTH,
            n_folds=N_TE_FOLDS,
        )

    for c in NUM_as_CAT:
        if c in X_tr.columns:
            add_cuml_te(
                X_tr,
                X_va,
                X_test,
                X_tr_g,
                X_va_g,
                X_test_g,
                y_tr_g,
                cols=(c,),
                out_col=c,
                fold_ids_g=fold_ids_g,
                seed=te_seed,
                smooth=TE_SMOOTH,
                n_folds=N_TE_FOLDS,
            )

    for cols in BIGRAM_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_cuml_te(
                X_tr,
                X_va,
                X_test,
                X_tr_g,
                X_va_g,
                X_test_g,
                y_tr_g,
                cols=cols,
                out_col="te2__" + "__".join(cols),
                fold_ids_g=fold_ids_g,
                seed=te_seed,
                smooth=TE_SMOOTH,
                n_folds=N_TE_FOLDS,
            )

    print("After TE:", X_tr.shape, X_va.shape, X_test.shape)

    fold_va_preds = np.zeros(len(X_va), dtype=np.float32)
    fold_test_preds = np.zeros(len(X_test), dtype=np.float32)

    for s, seed in enumerate(SEEDS):
        model_seed = seed + fold * 100

        print(f"  Seed {s + 1}/{len(SEEDS)}: {model_seed}")

        model = XGBClassifier(
            n_estimators=10000,
            learning_rate=0.03,

            grow_policy="lossguide",
            max_depth=0,
            max_leaves=LOSSGUIDE_MAX_LEAVES,

            min_child_weight=5,
            subsample=0.8,
            colsample_bytree=0.8,

            reg_alpha=0.0,
            reg_lambda=2.0,

            objective="binary:logistic",
            eval_metric="auc",
            random_state=model_seed,

            tree_method="hist",
            device="cuda",
            n_jobs=-1,
            enable_categorical=True,

            early_stopping_rounds=200,
        )

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=200,
        )

        va_pred = model.predict_proba(X_va)[:, 1].astype(np.float32)
        test_pred = model.predict_proba(X_test)[:, 1].astype(np.float32)

        fold_va_preds += va_pred / len(SEEDS)
        fold_test_preds += test_pred / len(SEEDS)

        seed_auc = roc_auc_score(y_va, va_pred)
        print(f"    Seed AUC: {seed_auc:.6f}")

    oof_preds[va_idx] = fold_va_preds
    test_preds += fold_test_preds / N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_va_preds)
    print(f"Fold {fold + 1} Ensemble AUC: {fold_auc:.6f}")

cv_auc = roc_auc_score(train[TARGET], oof_preds)
print(f"\nOOF AUC: {cv_auc:.6f}")

In [ ]:
pd.DataFrame({'id': train.id, TARGET: oof_preds}).to_csv(f'oof_xgb_{cv_auc}.csv', index=False)
pd.DataFrame({'id': test.id, TARGET: test_preds}).to_csv(f'test_xgb_{cv_auc}.csv', index=False)